<a href="https://colab.research.google.com/github/teoalcdor/trabajo_iae/blob/main/nanonets_ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nanonets-OCR-s:

Prueba de uso de Nanonets-OCR-s, que podemos encontrar [aquí](https://huggingface.co/nanonets/Nanonets-OCR-s). En principio parece algo lento a la hora procesar cada página, pero es muy preciso a la hora de extraer texto de imágenes y tablas y merece consideración.

## Entorno Virtual

Por si se desea acceder a Drive más tarde, lo montamos antes de crear y usar el entorno virtual:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Creamos un entorno virutual de Conda. Los paquetes que vienen pre-instalados en Colab, con sus correspondientes versiones, pueden dar problemas al instalar los requisitos, pero buscamos aprovechar las instancias de GPU y TPU de Colab.

In [ ]:
%env PYTHONPATH=
!pip install virtualenv
!virtualenv myenv
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda install -q -y --prefix /usr/local python=3.11.13 ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 36.6 MB/s eta 0:00:00
created virtual environment CPython3.11.13.final.0-64 in 331ms
  creator CPython3Posix(dest=/content/myenv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==25.1.1, setuptools==80.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
--2025-07-17 06:49:40--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 159476510 (152M) [application/octet-stream]
Saving to:

Activamos el entorno y pasamos a usarlo:

In [ ]:
import sys
import os
os.environ['CONDA_PREFIX'] = '/usr/local/envs/myenv'
sys.path.append('/usr/local/lib/python3.11.13/site-packages/')

Comprobamos que, efectivamente, no estamos usando el entorno proporcionado por Colab:

In [ ]:
!pip list

Package                  Version
------------------------ ---------
anaconda-anon-usage      0.7.1
annotated-types          0.6.0
archspec                 0.2.3
boltons                  25.0.0
brotlicffi               1.0.9.2
certifi                  2025.7.14
cffi                     1.17.1
charset-normalizer       3.3.2
conda                    25.5.1
conda-anaconda-telemetry 0.2.0
conda-anaconda-tos       0.2.0
conda-content-trust      0.2.0
conda-libmamba-solver    25.4.0
conda-package-handling   2.4.0
conda_package_streaming  0.12.0
cryptography             45.0.3
distro                   1.9.0
frozendict               2.4.2
idna                     3.7
jsonpatch                1.33
jsonpointer              2.1
libmambapy               2.0.5
markdown-it-py           2.2.0
mdurl                    0.1.0
menuinst                 2.3.0
packaging                24.2
pip                      25.1
platformdirs             4.3.7
pluggy                   1.5.0
pycosat                  0.6

## Descarga y Uso de Nanonets-OCR-s

Instalamos los requerimientos extra de Nanonets-OCR-s en Colab:

In [ ]:
!pip install --upgrade pip wheel setuptools packaging ninja
!pip install flash-attn --no-build-isolation


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 37.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, w

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 32.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 124.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 151.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 47.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 178.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 87.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 147.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 MB 149.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207

Descargamos y usamos el modelo:

In [ ]:
import torch
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText

In [ ]:
model_path = "nanonets/Nanonets-OCR-s"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2"
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_path)
processor = AutoProcessor.from_pretrained(model_path, use_fast=True)


def ocr_page_with_nanonets_s(image_path, model, processor, max_new_tokens=4096):
    prompt = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""
    image = Image.open(image_path)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]

    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return output_text[0]

cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
image_path = "/content/img_3.jpg"
result = ocr_page_with_nanonets_s(image_path, model, processor, max_new_tokens=15000)
print(result)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


el **RD1178/2023 de 27 de diciembre**. En resumen, los valores de ayudas establecidos son los siguientes:

<table>
  <tr>
    <td colspan="3"><strong>Gran empresa</strong><strong>Mediana empresa</strong><strong>Pequeña empresa</strong></td>
  </tr>
  <tr>
    <td><strong>Programa 1 y 2</strong></td>
    <td>15 % - 30 %</td>
    <td>25 % - 40 %</td>
    <td>35 % - 50 %</td>
  </tr>
  <tr>
    <td>Instalación autoconsumo</td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
  <tr>
    <td><strong>Programa 1, 2 y 3</strong></td>
    <td>30 %</td>
    <td>40 %</td>
    <td>50 %</td>
  </tr>
  <tr>
    <td>Instalación almacenamiento</td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
</table>

**Porcentaje ayuda sobre coste subvencionable**

El sector servicios y otros sectores productivos contarán con incentivos para instalaciones de autoconsumo con energía solar fotovoltaica y eólica que oscilan entre el 15% y el 50% en función de la tecnología, del tamaño de la empresa y de la potenc